# جست‌وجوی خودکار مقالات برای مرور نظام‌مند (OpenAlex)

1. ده عبارت را در OpenAlex جست‌وجو می‌کند.
2. تعداد کل نتایج هر عبارت و تعداد رکوردهای دریافت‌شده را ثبت می‌کند.
3. حداکثر ۲۰۰ رکورد برتر هر عبارت را (دو صفحهٔ ۱۰۰تایی) دریافت می‌کند.
4. عنوان، نویسندگان، سال، DOI، چکیده، نشریه/همایش و لینک‌ها را استخراج می‌کند.
5. رکوردها را ادغام و بر اساس DOI، شناسه OpenAlex و سپس عنوان حذف تکراری می‌کند.
6. فایل‌های CSV، فایل Excel، قالب غربالگری انسانی و گزارش جست‌وجو را می‌سازد و در قالب ZIP ارائه می‌دهد.

**نکته روش‌شناختی:** این یک جست‌وجوی تکمیلی جدید در OpenAlex است؛ آن را با جست‌وجوهای تاریخی Google Scholar، Semantic Scholar، arXiv و ACL Anthology یکی نگیرید. تعداد کل نتایج API با تعداد رکوردهای دریافت‌شده و تعداد رکوردهای یکتا متفاوت است. غربالگری نهایی باید انسانی باشد و PRISMA نهایی به شمارش واقعی همه منابع نیاز دارد.

نیازی به نصب Python روی رایانه نیست؛ در Colab اجرا کنید.

## ۱) نصب کتابخانه‌ها و تنظیمات

In [ ]:
!pip -q install pandas requests openpyxl

import re, time, zipfile, requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
# OUT = Path("/content/openalex_review_search")
OUT = Path("OpenAlex_Results")
# OUT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

START_DATE = "2023-01-01"
END_DATE = "2026-11-13"
CAP_PER_QUERY = 200
PER_PAGE = 100
API = "https://api.openalex.org/works"
RUN_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

QUERIES = [
    ("Q1", 'LLM knowledge distillation graph'),
    ("Q2", 'large language model graph neural network distillation'),
    ("Q3", 'language model graph knowledge transfer'),
    ("Q4", 'pretrained language model graph knowledge distillation'),
    ("Q5", 'language model teacher graph student'),
    ("Q6", 'LLM-guided graph knowledge transfer'),
    ("Q7", 'LLM soft-label distillation graph'),
    ("Q8", 'LLM rationale distillation graph learning'),
    ("Q9", 'temporal graph knowledge distillation language model'),
    ("Q10", '(LLM OR "large language model" OR PLM OR "pretrained language model") AND (distillation OR distilled OR "knowledge transfer") AND (graph OR GNN OR "graph neural network")'),
]
print("آماده اجرا؛ تعداد عبارت‌ها:", len(QUERIES))


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


آماده اجرا؛ تعداد عبارت‌ها: 10


## ۲) توابع دریافت، استخراج چکیده و نرمال‌سازی

In [2]:
def api_get(params, retries=5):
    headers = {"User-Agent": "AcademicLiteratureReviewSearch/1.0"}
    for attempt in range(retries):
        try:
            r = requests.get(API, params=params, headers=headers, timeout=60)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(2**attempt, 30))
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(min(2**attempt, 30))
    raise RuntimeError("OpenAlex پس از چند تلاش پاسخ نداد.")

def abstract_text(index):
    if not isinstance(index, dict):
        return ""
    pos = {}
    for word, indexes in index.items():
        for i in indexes:
            pos[int(i)] = word
    return " ".join(pos[i] for i in sorted(pos))

def norm_doi(x):
    x = (x or "").strip().lower()
    return re.sub(r"^https?://(dx\.)?doi\.org/", "", x)

def norm_title(x):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", (x or "").lower())).strip()

def unpack(work, qid, query, page):
    authors = []
    for a in work.get("authorships", []) or []:
        name = (a.get("author") or {}).get("display_name")
        if name: authors.append(name)
    loc = work.get("primary_location") or {}
    source = loc.get("source") or {}
    oa = work.get("open_access") or {}
    best = work.get("best_oa_location") or {}
    return {
        "openalex_id": work.get("id", ""),
        "doi": norm_doi(work.get("doi", "")),
        "title": work.get("title", "") or "",
        "publication_year": work.get("publication_year"),
        "publication_date": work.get("publication_date", ""),
        "authors": "; ".join(authors),
        "abstract": abstract_text(work.get("abstract_inverted_index")),
        "document_type": work.get("type", ""),
        "venue": source.get("display_name", "") or "",
        "landing_page_url": loc.get("landing_page_url", "") or "",
        "best_oa_url": best.get("landing_page_url", "") or "",
        "is_oa": oa.get("is_oa"),
        "oa_status": oa.get("oa_status", ""),
        "cited_by_count": work.get("cited_by_count"),
        "query_id": qid, "query_text": query, "retrieved_page": page
    }

## ۳) اجرای جست‌وجوها
برای هر عبارت ابتدا `meta.count` ثبت می‌شود و سپس حداکثر دو صفحه دریافت خواهد شد.

In [3]:
records, logs = [], []
for qid, query in QUERIES:
    print(f"در حال جست‌وجوی {qid}: {query}")
    params = {
        "search": query,
        "filter": f"from_publication_date:{START_DATE},to_publication_date:{END_DATE}",
        "per-page": PER_PAGE, "page": 1
    }
    first = api_get(params)
    total = (first.get("meta") or {}).get("count", 0)
    fetched = 0
    pages = 1 if total <= PER_PAGE else 2
    for page in range(1, pages + 1):
        data = first if page == 1 else api_get({**params, "page": page})
        for w in data.get("results", []) or []:
            if fetched >= CAP_PER_QUERY: break
            records.append(unpack(w, qid, query, page))
            fetched += 1
        time.sleep(0.15)
    logs.append({
        "query_id": qid, "query_text": query, "source": "OpenAlex Works API",
        "run_timestamp_utc": RUN_UTC, "date_from": START_DATE, "date_to": END_DATE,
        "total_hits_reported_by_api": total, "retrieval_cap": CAP_PER_QUERY,
        "records_retrieved": fetched, "pages_retrieved": pages, "status": "completed"
    })
    print(f"  کل نتایج API: {total:,} | دریافت‌شده: {fetched:,}")

raw = pd.DataFrame(records)
log = pd.DataFrame(logs)
print("مجموع رکوردهای خام:", len(raw))
display(log)

در حال جست‌وجوی Q1: "LLM knowledge distillation graph"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q2: "large language model graph neural network distillation"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q3: "language model graph knowledge transfer"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q4: "pretrained language model graph knowledge distillation"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q5: "language model teacher graph student"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q6: "LLM-guided graph knowledge transfer"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q7: "LLM soft-label distillation graph"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q8: "LLM rationale distillation graph learning"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q9: "temporal graph knowledge distillation language model"
  کل نتایج API: 0 | دریافت‌شده: 0
در حال جست‌وجوی Q10: (LLM OR "large language model" OR PLM OR "pretrained language model") AND (distillation OR

,query_id,query_text,source,run_timestamp_utc,date_from,date_to,total_hits_reported_by_api,retrieval_cap,records_retrieved,pages_retrieved,status
0,Q1,"""LLM knowledge distillation graph""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
1,Q2,"""large language model graph neural network distillation""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
2,Q3,"""language model graph knowledge transfer""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
3,Q4,"""pretrained language model graph knowledge distillation""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
4,Q5,"""language model teacher graph student""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
5,Q6,"""LLM-guided graph knowledge transfer""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
6,Q7,"""LLM soft-label distillation graph""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
7,Q8,"""LLM rationale distillation graph learning""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
8,Q9,"""temporal graph knowledge distillation language model""",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,0,200,0,1,completed
9,Q10,"(LLM OR ""large language model"" OR PLM OR ""pretrained language model"") AND (distillation OR distilled OR ""knowledge t...",OpenAlex Works API,2026-11-13T09:26:57+00:00,2023-01-01,2026-11-13,46003,200,200,2,completed


## ۴) ادغام و حذف تکراری‌ها
اولویت تشخیص تکراری: DOI، شناسه OpenAlex، عنوان نرمال‌شده.

In [4]:
def deduplicate(df):
    if df.empty:
        return df.copy(), pd.DataFrame()
    d = df.copy()
    d["title_norm"] = d["title"].map(norm_title)
    def key(row):
        if row["doi"]: return "doi:" + row["doi"]
        if row["openalex_id"]: return "openalex:" + row["openalex_id"]
        return "title:" + row["title_norm"]
    d["dedup_key"] = d.apply(key, axis=1)
    keep, dup = [], []
    for k, g in d.groupby("dedup_key", sort=False):
        g = g.copy()
        g["_score"] = (g["abstract"].fillna("").str.len()
                       + 2*g["doi"].fillna("").astype(str).str.len()
                       + g["authors"].fillna("").str.len()
                       + g["venue"].fillna("").str.len())
        g = g.sort_values("_score", ascending=False)
        row = g.iloc[0].drop(labels=["_score"])
        row["matched_queries"] = "; ".join(sorted(set(g["query_id"].dropna())))
        row["number_of_query_hits"] = len(g)
        keep.append(row)
        for _, x in g.iloc[1:].iterrows():
            dup.append({
                "dedup_key": k, "kept_title": row["title"],
                "duplicate_title": x["title"], "doi": x["doi"],
                "openalex_id": x["openalex_id"],
                "duplicate_query_id": x["query_id"],
                "duplicate_query_text": x["query_text"]
            })
    unique = pd.DataFrame(keep).drop(columns=["title_norm", "dedup_key"], errors="ignore")
    return unique, pd.DataFrame(dup)

unique, duplicates = deduplicate(raw)
print(f"خام: {len(raw):,} | یکتا: {len(unique):,} | تکراری حذف‌شده: {len(raw)-len(unique):,}")
display(unique[["title", "publication_year", "doi", "matched_queries"]].head(10))

خام: 200 | یکتا: 200 | تکراری حذف‌شده: 0


,title,publication_year,doi,matched_queries
0,Exploring the Potential of Large Language Models (LLMs)in Learning on Graphs,2024,10.1145/3655103.3655110,Q10
1,Parameter-Efficient Fine-Tuning Methods for Pretrained Language Models: A Critical Review and Assessment,2026,10.1109/tpami.2026.3657354,Q10
2,Large Language Model Meets Graph Neural Network in Knowledge Distillation,2025,10.1609/aaai.v39i16.33901,Q10
3,Large Language Models in Healthcare and Medical Domain: A Review,2024,10.3390/informatics11030057,Q10
4,GraphGPT: Graph Instruction Tuning for Large Language Models,2024,10.1145/3626772.3657775,Q10
5,LLM and GNN Are Complementary: Distilling LLM for Multimodal Graph Learning,2025,10.1109/bigdata66926.2025.11402034,Q10
6,Can Large Language Models Improve the Adversarial Robustness of Graph Neural Networks?,2025,10.1145/3690624.3709256,Q10
7,A Survey of Large Language Models for Graphs,2024,10.1145/3637528.3671460,Q10
8,A Survey of Graph Meets Large Language Model: Progress and Future Directions,2024,10.24963/ijcai.2024/898,Q10
9,Large Language Models on Graphs: A Comprehensive Survey,2024,10.1109/tkde.2024.3469578,Q10


## ۵) فایل آماده ASReview و قالب غربالگری انسانی

In [5]:
cols = ["title", "abstract", "authors", "publication_year", "doi",
        "openalex_id", "venue", "landing_page_url", "matched_queries"]
asreview = unique.reindex(columns=cols).copy()
screening = asreview.copy()
for col in ["title_abstract_decision", "title_abstract_exclusion_reason",
            "full_text_retrieved", "full_text_decision",
            "full_text_exclusion_reason", "reviewer", "screening_notes"]:
    screening[col] = ""
display(asreview.head())

,title,abstract,authors,publication_year,doi,openalex_id,venue,landing_page_url,matched_queries
0,Exploring the Potential of Large Language Models (LLMs)in Learning on Graphs,Learning on Graphs has attracted immense attention due to its wide real-world applications. The most popular pipelin...,Zhikai Chen; Haitao Mao; Hang Li; Wei Jin; Hongzhi Wen; Xiaochi Wei; Shuaiqiang Wang; Dawei Yin; Wenqi Fan; Hui Liu;...,2024,10.1145/3655103.3655110,https://openalex.org/W4393277515,ACM SIGKDD Explorations Newsletter,https://doi.org/10.1145/3655103.3655110,Q10
1,Parameter-Efficient Fine-Tuning Methods for Pretrained Language Models: A Critical Review and Assessment,"With the continuous growth in the number of parameters of the Transformer-based pretrained language models (PLMs), p...",Lingling Xu; Haoran Xie; S. Joe Qin; Xiaohui Tao; Fu Lee Wang,2026,10.1109/tpami.2026.3657354,https://openalex.org/W7125799258,IEEE Transactions on Pattern Analysis and Machine Intelligence,https://doi.org/10.1109/tpami.2026.3657354,Q10
2,Large Language Model Meets Graph Neural Network in Knowledge Distillation,"While Large Language Models (LLMs) show promise for Text-Attributed Graphs (TAGs) learning, their deployment is hind...",Shengxiang Hu; Guobing Zou; Yang Song; Shiyi Lin; Yanglan Gan; Bofeng Zhang; Yixin Chen,2025,10.1609/aaai.v39i16.33901,https://openalex.org/W4409364401,Proceedings of the AAAI Conference on Artificial Intelligence,https://doi.org/10.1609/aaai.v39i16.33901,Q10
3,Large Language Models in Healthcare and Medical Domain: A Review,The deployment of large language models (LLMs) within the healthcare sector has sparked both enthusiasm and apprehen...,Zabir Al Nazi; Wei Peng,2024,10.3390/informatics11030057,https://openalex.org/W4401386758,Informatics,https://doi.org/10.3390/informatics11030057,Q10
4,GraphGPT: Graph Instruction Tuning for Large Language Models,Graph Neural Networks (GNNs) have evolved to understand graph structures through recursive exchanges and aggregation...,Jiabin Tang; Yuhao Yang; Wei Wei; Lei Shi; Lixin Su; Suqi Cheng; Dawei Yin; Chao Huang,2024,10.1145/3626772.3657775,https://openalex.org/W4400524696,International ACM SIGIR Conference on Research and Development in Information Retrieval (SIGIR),https://doi.org/10.1145/3626772.3657775,Q10


## ۶) شمارش اولیه PRISMA (فقط OpenAlex)
سلول فقط شمارش‌های محاسبه‌شده از داده‌های دریافت‌شده را پر می‌کند. خانه‌های مراحل غربالگری باید پس از انجام غربالگری انسانی تکمیل شوند.

In [6]:
n_raw, n_unique = len(raw), len(unique)
prisma = pd.DataFrame([
    ["رکوردهای دریافت‌شده از OpenAlex پیش از حذف تکراری", n_raw],
    ["تکراری‌های حذف‌شده در مجموعه OpenAlex", n_raw-n_unique],
    ["رکوردهای یکتای OpenAlex برای غربالگری", n_unique],
    ["رکوردهای بررسی‌شده در عنوان/چکیده", ""],
    ["رکوردهای حذف‌شده در عنوان/چکیده", ""],
    ["متن کامل ارزیابی‌شده", ""],
    ["متن کامل حذف‌شده", ""],
    ["مطالعات واردشده در مرور نهایی", ""],
], columns=["مرحله", "تعداد"])
display(prisma)
print("این جدول فقط جست‌وجوی تکمیلی OpenAlex را پوشش می‌دهد، نه منابع تاریخی دیگر.")

,مرحله,تعداد
0,رکوردهای دریافت‌شده از OpenAlex پیش از حذف تکراری,200
1,تکراری‌های حذف‌شده در مجموعه OpenAlex,0
2,رکوردهای یکتای OpenAlex برای غربالگری,200
3,رکوردهای بررسی‌شده در عنوان/چکیده,
4,رکوردهای حذف‌شده در عنوان/چکیده,
5,متن کامل ارزیابی‌شده,
6,متن کامل حذف‌شده,
7,مطالعات واردشده در مرور نهایی,


این جدول فقط جست‌وجوی تکمیلی OpenAlex را پوشش می‌دهد، نه منابع تاریخی دیگر.


## ۷) ذخیره CSV، Excel و ZIP

In [7]:
# گزارش روش جست‌وجو
report = [
    "OPENALEX SUPPLEMENTARY SEARCH REPORT",
    f"Run timestamp UTC: {RUN_UTC}",
    f"Publication date range: {START_DATE} to {END_DATE}",
    f"Per-query retrieval cap: {CAP_PER_QUERY}",
    f"Raw retrieved records: {len(raw)}",
    f"Unique records: {len(unique)}",
    f"Duplicates removed: {len(raw)-len(unique)}",
    "",
    "This is a supplementary OpenAlex search, not a reconstruction of historical searches in Google Scholar, Semantic Scholar, arXiv, or ACL Anthology.",
    "API total_hits, records retrieved, and unique records are distinct counts.",
    "", "Queries:"
] + [f"{qid}: {q}" for qid, q in QUERIES]

files_out = {
    "OpenAlex_Search_Log.csv": log,
    "OpenAlex_Retrieved_Records_Raw.csv": raw,
    "OpenAlex_Unique_Records.csv": unique,
    "OpenAlex_Duplicate_Report.csv": duplicates,
    "ASReview_Import.csv": asreview,
    "Human_Screening_Template.csv": screening,
    "PRISMA_Counts_OpenAlex_Only.csv": prisma,
}
for name, frame in files_out.items():
    frame.to_csv(OUT/name, index=False, encoding="utf-8-sig")
(OUT/"OpenAlex_Search_Report.txt").write_text("\\n".join(report), encoding="utf-8")

xlsx = OUT/"OpenAlex_Search_and_Screening.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    for sheet, frame in [
        ("Search_Log", log), ("Raw_Records", raw), ("Unique_Records", unique),
        ("Duplicates", duplicates), ("ASReview_Import", asreview),
        ("Human_Screening", screening), ("PRISMA_Counts", prisma)
    ]:
        frame.to_excel(writer, sheet_name=sheet, index=False)

# zip_path = Path("/content/OpenAlex_Review_Search_Results.zip")
zip_path = Path("OpenAlex_Review_Search_Results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p, arcname=p.name)

print("فایل‌های خروجی:")
for p in sorted(OUT.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")
print("بسته ZIP:", zip_path)

فایل‌های خروجی:
ASReview_Import.csv (334,528 bytes)
Human_Screening_Template.csv (336,075 bytes)
OpenAlex_Duplicate_Report.csv (5 bytes)
OpenAlex_Retrieved_Records_Raw.csv (385,739 bytes)
OpenAlex_Search_and_Screening.xlsx (510,040 bytes)
OpenAlex_Search_Log.csv (1,686 bytes)
OpenAlex_Search_Report.txt (1,097 bytes)
OpenAlex_Unique_Records.csv (386,976 bytes)
PRISMA_Counts_OpenAlex_Only.csv (513 bytes)
بسته ZIP: OpenAlex_Review_Search_Results.zip


## ۸) دانلود بسته خروجی

In [9]:
# from google.colab import files
files.download(str(zip_path))

NameError: name 'files' is not defined

## ۹) راهنمای غربالگری و محدودیت‌ها

- فایل `ASReview_Import.csv` را در ASReview LAB وارد کنید. ASReview ترتیب بررسی را اولویت‌بندی می‌کند؛ تصمیم ورود/خروج باید توسط پژوهشگر گرفته شود.
- از `Human_Screening_Template.csv` برای ثبت تصمیم و دلیل حذف در هر مرحله استفاده کنید.
- معیار ورود مرور: LLM/PLM نقش معلم یا منبع دانش ممتاز داشته باشد و دانش آن به یک دانش‌آموز مستقل گراف‌محور منتقل شود؛ سازوکار انتقال و هدف آموزشی باید قابل شناسایی باشد.
- معیار خروج: استفاده صرفاً برای افزایش ویژگی/متن گراف، هم‌ترازی متقارن بدون نقش معلم، هم‌آموزی بدون معلم پایدار، استفاده صرفاً در زمان استنتاج، یا انتقال معکوس دانش از GNN به LLM.
- موارد مبهم را با `Maybe` مشخص و در متن کامل بررسی کنید.
- سقف ۲۰۰ رکورد برای هر عبارت اعمال شده است؛ اگر تعداد کل بیشتر باشد فقط ۲۰۰ نتیجهٔ اول دریافت می‌شود.
- چکیده همه رکوردها در OpenAlex موجود نیست.
- برای نمودار PRISMA نهایی، شمارش همه منابع، حذف تکراری‌های بین‌منبعی، غربالگری عنوان/چکیده و ارزیابی متن کامل لازم است. این دفترچه به‌تنهایی PRISMA نهایی را تولید نمی‌کند.
- این جست‌وجوی تکمیلی را در مقاله به‌صورت شفاف و با تاریخ اجرای واقعی گزارش کنید؛ آمار یا تاریخ تاریخیِ ثبت‌نشده را حدس نزنید.
